In [1]:
import sys

try:
    import tensorflow as tf
    current_tf_version = tf.__version__
    print(f"Current TensorFlow version detected: {current_tf_version}")
except ModuleNotFoundError:
    current_tf_version = None
    print("TensorFlow not found.")

target_tf_version = "2.21.0" # Updated to a newer available version

if current_tf_version != target_tf_version:
    if current_tf_version:
        print(f"Uninstalling TensorFlow version {current_tf_version} to switch to {target_tf_version}")
        # Uninstall current TensorFlow version
        !pip uninstall -y tensorflow
    else:
        print(f"Installing TensorFlow version {target_tf_version}")

    # Install the target TensorFlow version
    # Removed explicit numpy installation to let TensorFlow handle its dependencies
    !pip install tensorflow=={target_tf_version}

    # After installation, restart runtime
    print(f"TensorFlow {target_tf_version} installation complete.")
    print("Please click on the Runtime > Restart session and run all cells to use the newly installed version.")
else:
    print(f"TensorFlow {target_tf_version} is already installed.")

TensorFlow not found.
Installing TensorFlow version 2.21.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 572.6/572.6 MB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 96.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 327.1/327.1 kB 21.4 MB/s eta 0:00:00
  Attempting uninstall: protobuf
    Found existing installation: protobuf 5.29.6
    Uninstalling protobuf-5.29.6:
      Successfully uninstalled protobuf-5.29.6
  Attempting uninstall: h5py
    Found existing installation: h5py 3.16.0
    Uninstalling h5py-3.16.0:
      Successfully uninstalled h5py-3.16.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-cloud-discoveryengine 0.13.12 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<7.0.0,>=3.20.2, but you have protobuf 7.35.1 which is incompatible.
ydf 0.15.0 requires protobuf<7.0.0,>=5.29

TensorFlow 2.21.0 installation complete.
Please click on the Runtime > Restart session and run all cells to use the newly installed version.


In [2]:
import tensorflow as tf
import numpy as np
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense

In [6]:
l0 = Dense(units=1, input_shape=[1])
model = Sequential([l0])
model.compile(optimizer='sgd', loss='mean_squared_error')

xs = np.array([-1.0, 0.0, 1.0, 2.0, 3.0, 4.0], dtype=float)
ys = np.array([-3.0, -1.0, 1.0, 3.0, 5.0, 7.0], dtype=float)

model.fit(xs, ys, epochs=500)

print(model.predict(np.array([10.0])))
print("Here is what I learned: {}".format(l0.get_weights()))

Epoch 1/500
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 193ms/step - loss: 37.8733
Epoch 2/500
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step - loss: 30.1431
Epoch 3/500
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step - loss: 24.0543
Epoch 4/500
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step - loss: 19.2569
Epoch 5/500
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step - loss: 15.4758
Epoch 6/500
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step - loss: 12.4943
Epoch 7/500
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step - loss: 10.1420
Epoch 8/500
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step - loss: 8.2850
Epoch 9/500
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step - loss: 6.8177
Epoch 10/500
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step - loss: 5.6571
Epoch 11/500
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step - loss: 4.7380
Epoch 12/500
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step - loss: 4.0090
Epoch 13/500
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step - loss: 3.4297
Epoch 14/500
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step - loss: 2.9682
Epoch 15/500
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step - loss: 2.5996
Epoch 16/50

In [7]:
export_dir = 'saved_model/1'
tf.saved_model.save(model, export_dir)

In [8]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [9]:
# Convert the model.
converter = tf.lite.TFLiteConverter.from_saved_model(export_dir)
tflite_model = converter.convert()

In [10]:
import pathlib
tflite_model_file = pathlib.Path('model.tflite')
tflite_model_file.write_bytes(tflite_model)

1744

In [13]:
# Load TFLite model and allocate tensors.
interpreter = tf.lite.Interpreter(model_content=tflite_model)
interpreter.allocate_tensors()

# Get input and output tensors.
input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()
print(input_details)
print(output_details)

[{'name': 'serving_default_inputs:0', 'index': 0, 'shape': array([1, 1], dtype=int32), 'shape_signature': array([-1,  1], dtype=int32), 'dtype': <class 'numpy.float32'>, 'quantization': (0.0, 0), 'quantization_parameters': {'scales': array([], dtype=float32), 'zero_points': array([], dtype=int32), 'quantized_dimension': 0, 'block_size': 0}, 'sparsity_parameters': {}}]
[{'name': 'StatefulPartitionedCall:0', 'index': 6, 'shape': array([1, 1], dtype=int32), 'shape_signature': array([-1,  1], dtype=int32), 'dtype': <class 'numpy.float32'>, 'quantization': (0.0, 0), 'quantization_parameters': {'scales': array([], dtype=float32), 'zero_points': array([], dtype=int32), 'quantized_dimension': 0, 'block_size': 0}, 'sparsity_parameters': {}}]


In [17]:
to_predict = np.array([[10.0]], dtype=np.float32)
print(to_predict)
interpreter.set_tensor(input_details[0]['index'], to_predict)
interpreter.invoke()
tflite_results = interpreter.get_tensor(output_details[0]['index'])
print(tflite_results)

[[10.]]
[[-2.013036e+33]]


### Why is the TFLite prediction inaccurate?

The extreme inaccuracy (`[[-2.013036e+33]]`) is highly unusual and often points to an underlying issue with the environment or the model's initialization/loading rather than a subtle prediction error.

**The most probable cause is that the Colab runtime was not restarted after installing TensorFlow 2.21.0.** When you install new packages or change package versions using `!pip install`, the Python interpreter often needs to be restarted for those changes to fully take effect. If an older TensorFlow version was active during the TFLite conversion or inference, it could lead to this kind of corrupted output.

**Please ensure you have restarted the Colab runtime (Runtime > Restart session) and then run all cells from the beginning.** This will load the correct TensorFlow version and ensure consistency.

Let's also print the original Keras model's prediction for comparison and double-check the input details to confirm everything aligns as expected for the TFLite interpreter.

In [18]:
# Re-predict using the original Keras model for comparison
print("Original Keras model prediction for 10.0:")
print(model.predict(np.array([10.0])))

# Print debug information for TFLite input details and the prediction array
print("\nTFLite Input Details:")
print(input_details[0])
print("\nInput array 'to_predict' details:")
print(f"  Shape: {to_predict.shape}")
print(f"  Dtype: {to_predict.dtype}")

print("\nAfter restarting the runtime, please re-run all cells to see if the TFLite prediction is corrected.")

Original Keras model prediction for 10.0:
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 86ms/step
[[18.977951]]

TFLite Input Details:
{'name': 'serving_default_inputs:0', 'index': 0, 'shape': array([1, 1], dtype=int32), 'shape_signature': array([-1,  1], dtype=int32), 'dtype': <class 'numpy.float32'>, 'quantization': (0.0, 0), 'quantization_parameters': {'scales': array([], dtype=float32), 'zero_points': array([], dtype=int32), 'quantized_dimension': 0, 'block_size': 0}, 'sparsity_parameters': {}}

Input array 'to_predict' details:
  Shape: (1, 1)
  Dtype: float32

After restarting the runtime, please re-run all cells to see if the TFLite prediction is corrected.
